<a href="https://colab.research.google.com/github/juliaduboisas/plnEp1/blob/main/TF_IDF_Word_%2B_Linear_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview: TF-IDF Word + Linear SVM

* **TF-IDF**: Extrai características do texto. Atribui pesos maiores a palavras frequentes em um documento, mas raras no corpus geral.
* **Linear SVM**: Algoritmo de classificação que traça uma fronteira de decisão (hiperplano) para separar classes. É muito eficiente para dados de alta dimensionalidade, típicos de Processamento de Linguagem Natural (PLN).

In [ ]:
!git clone https://github.com/juliaduboisas/plnEp1.git

fatal: destination path 'plnEp1' already exists and is not an empty directory.


In [ ]:
!ls -la plnEp1

total 22324
drwxr-xr-x 4 root root     4096 Sep 29 14:28 .
drwxr-xr-x 1 root root     4096 Sep 29 14:28 ..
-rw-r--r-- 1 root root    77319 Sep 29 14:28 baselineModel.ipynb
-rw-r--r-- 1 root root   612715 Sep 29 14:28 bertimbauModel.ipynb
-rw-r--r-- 1 root root    72344 Sep 29 14:28 dataExploration.ipynb
-rw-r--r-- 1 root root   155194 Sep 29 14:28 ep1-enunciado.pdf
drwxr-xr-x 8 root root     4096 Sep 29 14:28 .git
-rw-r--r-- 1 root root     4628 Sep 29 14:28 .gitignore
drwxr-xr-x 2 root root     4096 Sep 29 14:28 pipelines
-rw-r--r-- 1 root root     1846 Sep 29 14:28 README.md
-rw-r--r-- 1 root root   631693 Sep 29 14:28 roBERTaModel.ipynb
-rw-r--r-- 1 root root   139620 Sep 29 14:28 sbertModel.ipynb
-rw-r--r-- 1 root root    75183 Sep 29 14:28 tfidf_sublinear.ipynb
-rw-r--r-- 1 root root 20960303 Sep 29 14:28 train.csv
-rw-r--r-- 1 root root    86684 Sep 29 14:28 useModel.ipynb


In [ ]:
import pandas as pd

# Carregando o dataset de treino
df_train = pd.read_csv('plnEp1/train.csv')

# Exibindo informações gerais sobre o dataset
print("Informações do dataset:")
display(df_train.info())

# Visualizando as primeiras linhas
print("\nPrimeiras linhas:")
display(df_train.head())

Informações do dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20092 entries, 0 to 20091
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   resp_text  20092 non-null  object
 1   clarity    20092 non-null  object
dtypes: object(2)
memory usage: 314.1+ KB


None


Primeiras linhas:


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Separando features (X) e target (y)
X = df_train['resp_text']
y = df_train['clarity']

# 2. Dividindo os dados (80% para treino e 20% para teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Exemplos para treino: {len(X_train)}")
print(f"Exemplos para teste: {len(X_test)}\n")

# 3. Construindo o Pipeline (TF-IDF + Linear SVM)
pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000)), # Limitamos a 10.000 palavras mais frequentes
    ('svm', LinearSVC(random_state=42))
])

# 4. Treinando o modelo
print("Treinando o modelo TF-IDF + Linear SVM...")
pipeline_svm.fit(X_train, y_train)
print("Treinamento concluído!\n")

# 5. Avaliando o modelo com os dados de teste
print("Avaliando performance no conjunto de teste:")
y_pred = pipeline_svm.predict(X_test)
print(classification_report(y_test, y_pred))

Exemplos para treino: 16073
Exemplos para teste: 4019

Treinando o modelo TF-IDF + Linear SVM...
Treinamento concluído!

Avaliando performance no conjunto de teste:
              precision    recall  f1-score   support

          c1       0.45      0.47      0.46      1262
        c234       0.38      0.35      0.36      1385
          c5       0.48      0.50      0.49      1372

    accuracy                           0.44      4019
   macro avg       0.44      0.44      0.44      4019
weighted avg       0.43      0.44      0.44      4019



In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Definindo os valores de C para testar
# O prefixo 'svm__' diz ao GridSearchCV que este parâmetro pertence ao passo 'svm' do nosso pipeline
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100]
}

# 2. Configurando o GridSearchCV
# Usaremos cv=5 (5-fold cross-validation) e otimizaremos para a métrica 'f1_macro'
grid_search = GridSearchCV(
    pipeline_svm,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

# 3. Executando a busca
print("Iniciando a busca pelo melhor parâmetro C (isso pode levar um tempinho)...")
grid_search.fit(X_train, y_train)

# 4. Mostrando os resultados
print("\n--- Resultados da Otimização ---")
print(f"Melhor parâmetro encontrado: {grid_search.best_params_}")
print(f"Melhor F1-score (Macro) na validação cruzada: {grid_search.best_score_:.4f}\n")

# 5. Avaliando o modelo otimizado no conjunto de teste (X_test)
print("Avaliando o modelo otimizado no conjunto de teste:")
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

Iniciando a busca pelo melhor parâmetro C (isso pode levar um tempinho)...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

--- Resultados da Otimização ---
Melhor parâmetro encontrado: {'svm__C': 0.1}
Melhor F1-score (Macro) na validação cruzada: 0.4476

Avaliando o modelo otimizado no conjunto de teste:
              precision    recall  f1-score   support

          c1       0.48      0.50      0.49      1262
        c234       0.41      0.34      0.37      1385
          c5       0.49      0.56      0.52      1372

    accuracy                           0.46      4019
   macro avg       0.46      0.47      0.46      4019
weighted avg       0.46      0.46      0.46      4019



In [ ]:
import nltk
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# 1. Baixando e configurando as stopwords em português
nltk.download('stopwords')
stop_words_pt = stopwords.words('portuguese')

# 2. Construindo o Pipeline com o Pré-processamento no TF-IDF
pipeline_svm_prep = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        stop_words=stop_words_pt,
        ngram_range=(1, 2),       # Extrai palavras sozinhas (unigramas) e pares de palavras (bigramas)
        strip_accents='unicode',  # Remove acentos das palavras
        lowercase=True            # Garante que tudo esteja em letras minúsculas
    )),
    ('svm', LinearSVC(C=0.1, random_state=42)) # Usando o C otimizado do GridSearchCV
])

# 3. Treinando o modelo atualizado
print("Treinando o modelo com Pré-processamento Avançado...")
pipeline_svm_prep.fit(X_train, y_train)
print("Treinamento concluído!\n")

# 4. Avaliando os resultados
print("Avaliando performance no conjunto de teste com pré-processamento:")
y_pred_prep = pipeline_svm_prep.predict(X_test)
print(classification_report(y_test, y_pred_prep))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ate', 'eramos', 'estao', 'estavamos', 'estiveramos', 'estivessemos', 'foramos', 'fossemos', 'ha', 'hao', 'houveramos', 'houverao', 'houveriamos', 'houvessemos', 'ja', 'nao', 'sao', 'sera', 'serao', 'seriamos', 'so', 'tambem', 'tera', 'terao', 'teriamos', 'tinhamos', 'tiveramos', 'tivessemos', 'voce', 'voces'] not in stop_words.
  warnings.warn(


Treinando o modelo com Pré-processamento Avançado...
Treinamento concluído!

Avaliando performance no conjunto de teste com pré-processamento:
              precision    recall  f1-score   support

          c1       0.49      0.50      0.49      1262
        c234       0.41      0.35      0.37      1385
          c5       0.49      0.55      0.52      1372

    accuracy                           0.46      4019
   macro avg       0.46      0.47      0.46      4019
weighted avg       0.46      0.46      0.46      4019



In [ ]:
import unicodedata

# 1. Função para remover acentos de uma string
def remove_accents(text):
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

# 2. Criar uma nova lista de stopwords sem acentos
stop_words_pt_no_accents = [remove_accents(word) for word in stop_words_pt]

# 3. Construindo o Pipeline com as stopwords corrigidas
pipeline_svm_prep_fixed = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        stop_words=stop_words_pt_no_accents, # Usando a lista sem acentos!
        ngram_range=(1, 2),
        strip_accents='unicode',
        lowercase=True
    )),
    ('svm', LinearSVC(C=0.1, random_state=42))
])

# 4. Treinando o modelo atualizado
print("Treinando o modelo com Stopwords corrigidas (sem acentos)...")
pipeline_svm_prep_fixed.fit(X_train, y_train)
print("Treinamento concluído! O aviso deve ter sumido.\n")

# 5. Avaliando os resultados
print("Avaliando performance no conjunto de teste:")
y_pred_prep_fixed = pipeline_svm_prep_fixed.predict(X_test)
print(classification_report(y_test, y_pred_prep_fixed))


Treinando o modelo com Stopwords corrigidas (sem acentos)...
Treinamento concluído! O aviso deve ter sumido.

Avaliando performance no conjunto de teste:
              precision    recall  f1-score   support

          c1       0.48      0.48      0.48      1262
        c234       0.41      0.35      0.38      1385
          c5       0.49      0.55      0.52      1372

    accuracy                           0.46      4019
   macro avg       0.46      0.46      0.46      4019
weighted avg       0.46      0.46      0.46      4019

